In [23]:
import pandas as pd
import ast
import json
df = pd.read_csv("/Users/yavuzlule/Desktop/bsc-relish/notebooks/relish_translated_gepeto.csv")

In [24]:
df.head()

,Unnamed: 0,recipe_id,source_id,title,text,ingredients,instructions,variations,page_number,recipe_number,...,indexed_cuisine,website,tei_data,chapter_number,book,notes,raw_title,id,LANGUAGE,translation
0,0,page_78,a_miscellany,How You Want to Make a Food of Hens,3 lb chicken 3/4 t anise in eggs 12 threads sa...,NaN,3 lb chicken 3/4 t anise in eggs 12 threads sa...,NaN,78.0,NaN,...,14th,NaN,NaN,NaN,NaN,NaN,NaN,a_miscellany,English,## Ginger-Saffron Chicken\n\n3 pounds chicken\...
1,1,page_112,a_miscellany,To Make Cheesecakes,"Take 12 quarts of milk warm from the cow, turn...",NaN,"Take 12 quarts of milk warm from the cow, turn...",NaN,112.0,NaN,...,17th,NaN,NaN,NaN,NaN,NaN,NaN,a_miscellany,English,Combine 12 quarts of warm milk with a generous...
2,2,page_129,a_miscellany,A Good Filling,4 large Granny Smith apples dough:1/2 cup flou...,NaN,4 large Granny Smith apples dough:1/2 cup flou...,NaN,129.0,NaN,...,14th,NaN,NaN,NaN,NaN,NaN,NaN,a_miscellany,English,## Apple Bake with Honey Glaze\n\n**Ingredient...
3,3,page_41,a_miscellany,Chicken Covered With Walnuts and Saffron,4.8 lbs chicken 1/2 t salt topping:\n1 c cilan...,NaN,4.8 lbs chicken 1/2 t salt topping:\n1 c cilan...,NaN,41.0,NaN,...,13th,NaN,NaN,NaN,NaN,NaN,NaN,a_miscellany,English,## Chicken with Walnut & Herb Meatballs\n\n**Y...
4,4,page_53,a_miscellany,Adas,1 c lentils 2 t dried sumac 2 T parsley (chopp...,NaN,1 c lentils 2 t dried sumac 2 T parsley (chopp...,NaN,53.0,NaN,...,15th c,NaN,NaN,NaN,NaN,NaN,NaN,a_miscellany,English,1 cup lentils\n2 teaspoons dried sumac\n2 tabl...


In [25]:
df['LANGUAGE'].value_counts()

LANGUAGE
Old German              2501
Catalan                 1056
English                  792
Italian                  386
Old French               376
Venetian/Italian         270
French                   203
Early Modern English     181
Middle Low German        102
Multiple                  60
German                    40
Old Danish                25
Latin                     17
Middle French             12
Middle Dutch               3
Name: count, dtype: int64

In [41]:
import pandas as pd
import json
import re

def to_sentence_case(s: str) -> str:
    if not isinstance(s, str) or not s:
        return s
    s = s.strip().lower()
    return re.sub(r'(^\s*\w|[.!?]\s*\w)', lambda m: m.group().upper(), s)

def clean_text(s: str) -> str:
    if not isinstance(s, str):
        return s

    # remove markdown headings, extra hashes
    s = re.sub(r"#+\s*", "", s)

    # remove unwanted unicode/odd spacing artifacts
    s = re.sub(r"[^\x00-\x7F]+", " ", s)   # drop non-ascii (optional)
    s = re.sub(r"\s+", " ", s).strip()

    # sentence case
    s = s.lower()
    s = re.sub(r"(^\w|[.!?\n]\s*\w)", lambda m: m.group().upper(), s)

    return s


def finalize_df(df):
    cols = ["title", "text", "translation", "LANGUAGE"]
    return df[[c for c in cols if c in df.columns]]

In [31]:

df_final = finalize_df(df)

df_final.head()

,title,text,translation,LANGUAGE
0,How You Want to Make a Food of Hens,3 lb chicken 3/4 t anise in eggs 12 threads sa...,## Ginger-Saffron Chicken\n\n3 pounds chicken\...,English
1,To Make Cheesecakes,"Take 12 quarts of milk warm from the cow, turn...",Combine 12 quarts of warm milk with a generous...,English
2,A Good Filling,4 large Granny Smith apples dough:1/2 cup flou...,## Apple Bake with Honey Glaze\n\n**Ingredient...,English
3,Chicken Covered With Walnuts and Saffron,4.8 lbs chicken 1/2 t salt topping:\n1 c cilan...,## Chicken with Walnut & Herb Meatballs\n\n**Y...,English
4,Adas,1 c lentils 2 t dried sumac 2 T parsley (chopp...,1 cup lentils\n2 teaspoons dried sumac\n2 tabl...,English


In [32]:
df_final["LANGUAGE"].value_counts()

LANGUAGE
Old German              2501
Catalan                 1056
English                  792
Italian                  386
Old French               376
Venetian/Italian         270
French                   203
Early Modern English     181
Middle Low German        102
Multiple                  60
German                    40
Old Danish                25
Latin                     17
Middle French             12
Middle Dutch               3
Name: count, dtype: int64

In [43]:
df = df_final

df["text"] = df["text"].apply(clean_text)
df["translation"] = df["translation"].apply(clean_text)
df["title"] = df["title"].apply(clean_text)


df.head()

,title,text,translation,LANGUAGE
0,How you want to make a food of hens,3 lb chicken 3/4 t anise in eggs 12 threads sa...,Ginger-saffron chicken 3 pounds chicken teaspo...,English
1,To make cheesecakes,"Take 12 quarts of milk warm from the cow, turn...",Combine 12 quarts of warm milk with a generous...,English
2,A good filling,4 large granny smith apples dough:1/2 cup flou...,Apple bake with honey glaze **ingredients:** *...,English
3,Chicken covered with walnuts and saffron,4.8 lbs chicken 1/2 t salt topping: 1 c cilant...,Chicken with walnut & herb meatballs **yields:...,English
4,Adas,1 c lentils 2 t dried sumac 2 t parsley (chopp...,1 cup lentils 2 teaspoons dried sumac 2 tables...,English


In [44]:
df.to_parquet("relish.parquet")

In [47]:
import pandas as pd

def chunk_text_words(text, chunk_size=256):
    if not isinstance(text, str):
        return []

    words = text.split()
    return [
        " ".join(words[i:i + chunk_size])
        for i in range(0, len(words), chunk_size)
    ]

def chunk_dataframe(df, chunk_size=256):
    df = df.copy()

    df["chunk_text"] = df["text"].apply(lambda x: chunk_text_words(x, chunk_size))

    df = df.explode("chunk_text", ignore_index=True)

    # recompute chunk_id per original row using grouping on stable key
    df["chunk_id"] = df.groupby(df["title"]).cumcount()

    return df

In [48]:
df_chunked = chunk_dataframe(df)
df_chunked.head()

,title,text,translation,LANGUAGE,chunk_text,chunk_id
0,How you want to make a food of hens,3 lb chicken 3/4 t anise in eggs 12 threads sa...,Ginger-saffron chicken 3 pounds chicken teaspo...,English,3 lb chicken 3/4 t anise in eggs 12 threads sa...,0.0
1,To make cheesecakes,"Take 12 quarts of milk warm from the cow, turn...",Combine 12 quarts of warm milk with a generous...,English,"Take 12 quarts of milk warm from the cow, turn...",0.0
2,To make cheesecakes,"Take 12 quarts of milk warm from the cow, turn...",Combine 12 quarts of warm milk with a generous...,English,70 minutes. Let cool 1 hour before serving.,1.0
3,A good filling,4 large granny smith apples dough:1/2 cup flou...,Apple bake with honey glaze **ingredients:** *...,English,4 large granny smith apples dough:1/2 cup flou...,0.0
4,Chicken covered with walnuts and saffron,4.8 lbs chicken 1/2 t salt topping: 1 c cilant...,Chicken with walnut & herb meatballs **yields:...,English,4.8 lbs chicken 1/2 t salt topping: 1 c cilant...,0.0


In [49]:
df_chunked.to_parquet("relish_chunked.parquet")